# VGSales Data Cleaning
Notebook này thực hiện việc kiểm tra, làm sạch và chuẩn hóa dữ liệu từ file `vgsales.csv`.
Các bước chính bao gồm: loại bỏ dòng thiếu năm phát hành, điền thông tin nhà phát hành bị thiếu, xử lý trùng lặp và tính toán lại tổng doanh thu để đảm bảo tính nhất quán.

## 1. Import Thư viện và Đọc Dữ liệu

In [ ]:
import pandas as pd
import numpy as np

# Đọc file CSV
df = pd.read_csv('vgsales.csv')

print(f"Kích thước dữ liệu ban đầu: {df.shape}")
display(df.head())

## 2. Kiểm tra Dữ liệu (Data Profiling)
Xem xét số lượng giá trị rỗng (Missing Values) và kiểu dữ liệu hiện tại.

In [ ]:
print("--- Thông tin Dữ liệu ---")
df.info()

print("\n--- Số lượng Missing Values ---")
display(df.isnull().sum())

## 3. Xử lý Dữ liệu Khuyết thiếu (Missing Values) và Ép kiểu
- Xóa bỏ các dòng không có năm phát hành (`Year`).
- Chuyển cột `Year` về dạng số nguyên (`int`).
- Với cột `Publisher`, điền giá trị `'Unknown'` (Không rõ) thay vì xóa bỏ.

In [ ]:
# Pandas có thể đọc chuỗi 'N/A' thành NaN. Chắc chắn chuyển đổi sang numeric để loại bỏ lỗi chuỗi.
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

# Xóa các dòng thiếu Year
initial_len = len(df)
df = df.dropna(subset=['Year'])
print(f"Đã xóa {initial_len - len(df)} dòng bị thiếu năm phát hành.")

# Ép kiểu Year về int
df['Year'] = df['Year'].astype(int)

# Xử lý Publisher trống
df['Publisher'] = df['Publisher'].fillna('Unknown')

print("\nSố lượng Missing Values sau khi xử lý:")
display(df.isnull().sum())

## 4. Kiểm tra Định dạng Chuỗi (Formatting Anomalies)
Kiểm tra xem các cột văn bản (`Name`, `Platform`, `Genre`, `Publisher`) có khoảng trắng thừa hay không, và xem các danh mục có bị lỗi chính tả không.

In [ ]:
string_cols = ['Name', 'Platform', 'Genre', 'Publisher']

for col in string_cols:
    # Xóa khoảng trắng thừa (leading/trailing spaces) để chuẩn hóa định dạng
    df[col] = df[col].astype(str).str.strip()

print("Các nền tảng (Platform) hiện có sau chuẩn hóa:")
print(sorted(df['Platform'].unique()))

print("\nCác thể loại (Genre) hiện có sau chuẩn hóa:")
print(sorted(df['Genre'].unique()))

print("\nKiểm tra dải năm (Year Range):")
print(f"Năm nhỏ nhất: {df['Year'].min()}, Năm lớn nhất: {df['Year'].max()}")
if df['Year'].max() > 2026:
    print("\nCẢNH BÁO: Có năm phát hành lớn hơn hiện tại, cần xem xét lại.")

## 5. Xử lý Trùng lặp (Duplicates)
Tìm và gộp (aggregate) các trường hợp cùng 1 game phát hành trên cùng 1 hệ máy nhiều lần (Name + Platform bị trùng).

In [ ]:
dups = df[df.duplicated(subset=['Name', 'Platform'], keep=False)].sort_values(by=['Name', 'Platform'])
print(f"Phát hiện {len(dups)} dòng liên quan đến trùng lặp (Game + Hệ máy).")
if not dups.empty:
    display(dups)
    
    # Gộp bằng cách cộng dồn doanh thu
    agg_funcs = {
        'Year': 'first',
        'Genre': 'first',
        'Publisher': 'first',
        'NA_Sales': 'sum',
        'EU_Sales': 'sum',
        'JP_Sales': 'sum',
        'Other_Sales': 'sum',
        'Global_Sales': 'sum'
    }
    df = df.groupby(['Name', 'Platform'], as_index=False).agg(agg_funcs)
    print("Đã gộp thành công các dòng trùng lặp.")

## 6. Tính toán lại Tổng Doanh thu (Data Integrity)
Để tránh sai số do quá trình nhập liệu hoặc làm tròn, ta tính lại `Global_Sales` bằng tổng các khu vực.

In [ ]:
df['Global_Sales'] = df['NA_Sales'] + df['EU_Sales'] + df['JP_Sales'] + df['Other_Sales']

# Sắp xếp lại dữ liệu theo thứ tự doanh thu toàn cầu giảm dần
df = df.sort_values('Global_Sales', ascending=False).reset_index(drop=True)

# Tạo lại cột Rank (Thứ hạng) cho chuẩn xác
df['Rank'] = df.index + 1

# Sắp xếp lại thứ tự cột như ban đầu
cols = ['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
df = df[cols]

display(df.head())

## 7. Lưu Dữ liệu đã làm sạch
Lưu vào file mới `vgsales_cleaned.csv`.

In [ ]:
output_file = 'vgsales_cleaned.csv'
df.to_csv(output_file, index=False)
print(f"Dữ liệu đã được làm sạch và lưu tại {output_file} với {len(df)} dòng.")